# Lab 05 — Entrenar, validar y probar sin fuga de datos

Práctica didáctica generada con IA, revisada el 26/10/2026. Usa solo la biblioteca estándar de Python 3;
no necesita GPU, conexión ni descargas. Ejecuta las celdas en orden en un lector de notebooks.

**Objetivo:** explicar cómo la pérdida produce un gradiente, distinguir entrenamiento, validación y prueba,
y comparar un clasificador con una línea base. Requisitos: regresión logística (Clase 06) y matriz de confusión
(Clase 21), PAO 2026-II. Las fórmulas siguen esas diapositivas; los datos son sintéticos.

Antes de ejecutar: predice si cambiar el umbral cambia los pesos. Al terminar tendrás que justificarlo.


In [ ]:
import math
import random
import sys

SEMILLA = 17
rng = random.Random(SEMILLA)
print("Python", sys.version.split()[0], "— semilla", SEMILLA)
# Etiqueta probabilística: los grupos se solapan; una separación perfecta no está garantizada.
def sigmoide(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    e = math.exp(z)
    return e / (1 + e)

datos = []
for _ in range(240):
    x = rng.uniform(-2, 2)
    y = int(rng.random() < sigmoide(1.5 * x - 0.3))
    datos.append((x, y))
rng.shuffle(datos)
entrenamiento, validacion, prueba = datos[:144], datos[144:192], datos[192:]
assert len(entrenamiento) + len(validacion) + len(prueba) == len(datos)
print("Tamaños:", len(entrenamiento), len(validacion), len(prueba))


## Entrena solo con entrenamiento

Usamos h(x) = sigmoide(w·x + b). El gradiente medio es promedio(h−y) para b y promedio((h−y)·x)
para w. Ambos se calculan con los parámetros anteriores. La pérdida se calcula de forma estable desde
el logit z: max(z,0) − y·z + log(1 + exp(−|z|)).

Predice el sentido de la primera actualización antes de consultar los parámetros.


In [ ]:
def costo(datos, w, b):
    return sum(max(w*x+b, 0) - y*(w*x+b) + math.log1p(math.exp(-abs(w*x+b)))
               for x, y in datos) / len(datos)

def entrenar(datos, tasa=0.2, pasos=200):
    w, b = 0.0, 0.0
    historial = [costo(datos, w, b)]
    for _ in range(pasos):
        errores = [(x, sigmoide(w*x+b) - y) for x, y in datos]
        dw = sum(x*e for x, e in errores) / len(datos)
        db = sum(e for _, e in errores) / len(datos)
        w, b = w - tasa*dw, b - tasa*db
        historial.append(costo(datos, w, b))
    return w, b, historial

w, b, historial = entrenar(entrenamiento)
assert all(math.isfinite(v) for v in historial)
assert historial[-1] < historial[0]
print("w, b:", round(w, 4), round(b, 4))
print("Costo inicial/final:", round(historial[0], 4), round(historial[-1], 4))


## Selecciona con validación y evalúa una vez con prueba

Esta política didáctica elige entre tres umbrales por F1 de validación. No es una recomendación universal:
en una aplicación real el costo de los errores debe justificar la métrica. Fijamos todas las decisiones
antes de abrir prueba. Una métrica con denominador cero se reporta como indefinida.


In [ ]:
def metricas(reales, predichas):
    assert len(reales) == len(predichas) and len(reales) > 0
    vp = sum(y == 1 and p == 1 for y, p in zip(reales, predichas))
    fp = sum(y == 0 and p == 1 for y, p in zip(reales, predichas))
    fn = sum(y == 1 and p == 0 for y, p in zip(reales, predichas))
    vn = len(reales) - vp - fp - fn
    def dividir(a, b):
        return a / b if b else None
    return dict(VP=vp, FP=fp, FN=fn, VN=vn,
                exactitud=(vp+vn)/len(reales), precision=dividir(vp,vp+fp),
                recuperacion=dividir(vp,vp+fn), f1=dividir(2*vp,2*vp+fp+fn))

def evaluar(datos, umbral):
    return metricas([y for _, y in datos], [int(sigmoide(w*x+b) >= umbral) for x, _ in datos])

candidatos = {t: evaluar(validacion, t) for t in (0.3, 0.5, 0.7)}
# Desempate declarado: primer umbral en el orden anterior.
umbral = max(candidatos, key=lambda t: candidatos[t]['f1'] if candidatos[t]['f1'] is not None else -1)
print("Validación:", candidatos)
print("Umbral elegido:", umbral)
resultado = evaluar(prueba, umbral)
# Línea base: clase mayoritaria calculada SOLO con entrenamiento.
mayoritaria = int(sum(y for _, y in entrenamiento) >= len(entrenamiento)/2)
base = metricas([y for _, y in prueba], [mayoritaria]*len(prueba))
print("Prueba:", resultado)
print("Línea base:", base)
assert sum(resultado[k] for k in ('VP','FP','FN','VN')) == len(prueba)
assert metricas([1, 1, 0, 0], [1, 0, 1, 0])['f1'] == 0.5
assert metricas([0, 0], [0, 0])['precision'] is None


## Interpreta antes de cambiar el código

1. ¿Por qué bajar el costo de entrenamiento no garantiza mejorar prueba?
2. Cambia la tasa y compara SOLO entrenamiento y validación. Si ya usaste prueba para tomar decisiones,
   deja de considerarla una estimación final independiente: necesitarías nuevos datos de prueba.
3. Calcula a mano precisión y recuperación a partir de los conteos impresos.
4. ¿Por qué la exactitud de la línea base puede ocultar que no encuentra positivos?
5. ¿Qué harías si varias filas pertenecieran a la misma persona? Deben permanecer en una única partición.

**Comprobación:** el umbral transforma probabilidades en clases; no cambia w ni b. El resultado de este
experimento solo describe estos datos sintéticos y esta partición. No valida una aplicación real ni
reemplaza la experimentación con los datos del proyecto.

Fuentes del curso: Clase 06 (regresión logística) y Clase 21 (matriz de confusión), PAO 2026-II.
